# Event Consumer & MITRE ATT&CK Classifier

## Purpose
This notebook is the **detection / enrichment layer** of the pipeline. It:

1. consumes raw events from Kafka asynchronously,
2. classifies each event against the **MITRE ATT&CK** framework with a small rule set,
3. emits OpenTelemetry spans to Jaeger so the whole producer → consumer flow shows up as one distributed trace,
4. appends classified records to `classified_packets.csv`.

## Tracing
We reconstruct the trace from the event_id (deterministic UUID-to-trace-id mapping). The consumer emits three spans:
* `consume_event` - top-level CONSUMER span
* `classify_event` - the MITRE rule evaluation
* `write_csv` - sink write

In [ ]:
!pip install -q kafka-python opentelemetry-sdk opentelemetry-exporter-otlp

In [ ]:
import json
import csv
import os
import random
from datetime import datetime

from kafka import KafkaConsumer

# OpenTelemetry imports
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import (
    OTLPSpanExporter,
)
from opentelemetry.trace import SpanKind, TraceFlags
from opentelemetry.trace import set_span_in_context
from opentelemetry.trace import SpanContext, NonRecordingSpan

# Configuration
KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
KAFKA_TOPIC = "raw-events"
KAFKA_GROUP_ID = "packet-classifier"

OTLP_ENDPOINT = "jaeger:4317"
SERVICE_NAME = "packet-classifier"

OUTPUT_CSV = "classified_packets.csv"

# Tracing setup
resource = Resource.create(
    {
        "service.name": SERVICE_NAME,
    }
)

trace.set_tracer_provider(TracerProvider(resource=resource))
tracer = trace.get_tracer(__name__)

otlp_exporter = OTLPSpanExporter(
    endpoint=OTLP_ENDPOINT,
    insecure=True,
)

span_processor = BatchSpanProcessor(otlp_exporter)
trace.get_tracer_provider().add_span_processor(span_processor)

# Kafka consumer
consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    group_id=KAFKA_GROUP_ID,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
)

print("Kafka consumer started")
print(f"Topic: {KAFKA_TOPIC}")
print(f"OTLP endpoint: {OTLP_ENDPOINT}")

# MITRE ATT&CK classification
def classify_event(event):
    """
    Simple rule-based MITRE ATT&CK classification.
    """

    if event["event_type"] == "process_start":
        cmd = (event.get("command_line") or "").lower()

        if "encodedcommand" in cmd:
            return "TA0002", "T1059.001"  # Execution / PowerShell
        if "cmd.exe" in cmd:
            return "TA0002", "T1059.003"  # Execution / Windows Command Shell
        return "TA0002", "T1059"

    if event["event_type"] == "user_login":
        if event.get("logon_type") == "failure":
            return "TA0006", "T1110"  # Credential Access / Brute Force
        return "TA0001", "T1078"      # Initial Access / Valid Accounts

    return "TA0000", "T0000"

# CSV initialization
file_exists = os.path.exists(OUTPUT_CSV)

csv_file = open(OUTPUT_CSV, "a", newline="")
writer = csv.DictWriter(
    csv_file,
    fieldnames=[
        "event_id",
        "timestamp",
        "user",
        "host",
        "source_ip",
        "event_type",
        "mitre_tactic",
        "mitre_technique",
    ],
)

if not file_exists:
    writer.writeheader()

# Main consume loop
for msg in consumer:
    event = msg.value
    event_id = event["event_id"]

    # Restore trace context from event_id
    trace_id = int(event_id.replace("-", ""), 16)

    parent_ctx = set_span_in_context(
        NonRecordingSpan(
            SpanContext(
                trace_id=trace_id,
                span_id=random.getrandbits(64),
                is_remote=True,
                trace_flags=TraceFlags(TraceFlags.SAMPLED),
                trace_state={},
            )
        )
    )

    with tracer.start_as_current_span(
        "consume_event",
        context=parent_ctx,
        kind=SpanKind.CONSUMER,
    ) as span:

        span.set_attribute("event.id", event_id)
        span.set_attribute("event.type", event["event_type"])
        span.set_attribute("host.name", event["host"])
        span.set_attribute("user.name", event["user"])

        with tracer.start_as_current_span("classify_event"):
            tactic, technique = classify_event(event)

        span.set_attribute("mitre.tactic", tactic)
        span.set_attribute("mitre.technique", technique)

        record = {
            "event_id": event_id,
            "timestamp": event["timestamp"],
            "user": event["user"],
            "host": event["host"],
            "source_ip": event["source_ip"],
            "event_type": event["event_type"],
            "mitre_tactic": tactic,
            "mitre_technique": technique,
        }

        with tracer.start_as_current_span("write_csv"):
            writer.writerow(record)
            csv_file.flush()

        print(
            f"Processed {event_id} → {tactic} / {technique}"
        )